In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
df = pd.read_csv('/kaggle/input/q1-ka-ai-2026/Q1_data.csv')


In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:

df.info()

# Since after inspectening the data set using df.info(), I have notice that there are some missing values..

In [ ]:
# Task 4: Write your code here:

df.describe()

# After describing the dataset I have noticed that there are outliers in the 'Delivery_Time' column.

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

df['Delivery_Time'].value_counts()

In [ ]:
plt.figure(figsize= (10, 5))
plt.hist(df['Delivery_Time'].value_counts(), bins= 20, color= 'purple', edgecolor= 'black')
plt.ylabel('Frequency')
plt.xlabel('delivery_time')
plt.title('Target distribution')

In [ ]:
# Task 1: Write your code here:

df = df.drop('Order_ID', axis= 1)
df.head()

In [ ]:
df.columns

In [ ]:
# Task 2: Write your code here:

missing_values = df.isnull().sum()
missing_values

# There is an obsirvation:
# Since our target 'Delivery_Time' has 106 missing values, I should drop these samples; becuase our target must not be null values.

In [ ]:
df = df.dropna(subset=['Delivery_Time', 'Weather', 'Traffic_Level', 'Time_of_Day']) # dropping smaples that contains NA values in target column.
df.head()

In [ ]:
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean())

missing_values = df.isnull().sum()
print(f'The number of missing values are: {missing_values.sum()}')

In [ ]:
# Task 3: Write your code here:

duplicated_values = df.duplicated()
print(f'The number of duplicated values are: {duplicated_values.sum()}')

In [ ]:
print('Handing duplicated..')
df = df.drop_duplicates()
print(f'The number of duplicated values now are: {df.duplicated().sum()}')

len(df)

In [ ]:
# Task 4: Write your code here:
# Encode categorical variables if needed (Bonus if used One Hot Encoding)

from sklearn.preprocessing import OneHotEncoder, LabelEncoder

# I will use the both types of encoding, Why? Because there are two columns that are ordinal, 'Traffic_Level', 'Time_of_Day' and need to be labeled encoded, and the others will be OneHotEncoded.

ordered_categories = ['Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Weather']

le = LabelEncoder()
for col in ordered_categories:
  df[col] = le.fit_transform(df[col])

df.head()


In [ ]:
# Task 5: Write your code here:

from sklearn.preprocessing import StandardScaler

selected_features = df.drop('Delivery_Time', axis= 1).columns
standardscaler = StandardScaler()

df[selected_features] = standardscaler.fit_transform(df[selected_features])
df.head()

In [ ]:
# Task 6: Write your code here:

# No need for checking the target, becuase it is a predicting model.

In [ ]:
# Task 1: Write your code here:

X = df.drop('Delivery_Time', axis= 1).copy()
y = df['Delivery_Time']

y

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

mae_scores = []
rmse_scores = []
y_pred = []

for train_idx, val_idx in kfold.split(X):
    X_fold_train, X_fold_val = X.iloc[train_idx], X.iloc[val_idx]
    y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)
    y_pred.extend(y_fold_pred)
    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)

print(f"MAE: {mae_scores.mean():,.2f}")


In [ ]:
# Task 1: Write your code here:

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

plt.figure(figsize= (10, 5))
plt.hist(y_pred, bins= 20, color= 'purple', edgecolor= 'black')
plt.ylabel('Frequency')
plt.xlabel('delivery_time')
plt.title('Target distribution')

In [ ]:
%pip install catboost

In [ ]:
!pip install catboost

In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegresoor

models = {
    "Random Forest": RandomForestRegressor(
      n_estimators=320,  # Number of trees
      max_depth=4
  ),
    "CatBoost": CatBoostRegresoor(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
}

all_results = {}

for name in models:
  all_results[name] = {'MAE': []}

n_splits = 5

skf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():

    print(f"Training {model_name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)


    mae = mean_absolute_error(y_pred, y_test)
    all_results[model_name]['MAE'].append(mae)

